# 1. Introducao

Trilha C: **nnPU (Non-Negative Positive-Unlabeled Learning)** como trilha principal da familia
**PU explicito / risk-based**.

Pergunta metodologica central:
- como ordenar casos pela propensao latente positiva quando so observamos o rotulo proxy `s`?

Ponto central da trilha:
- `s` e rotulo observado (proxy), nao `y` latente real.
- nnPU modela risco PU explicitamente, usando `pi_p = P(y=1)` como hipotese operacional.
- a avaliacao empirica continua baseada em `s`, por limitacao do dado observado.

Estrutura interna desta notebook:
- **C1 - prior fixo**: execucao principal com `PI_P_MAIN`;
- **C2 - grid de prior**: sensibilidade obrigatoria e destacada;
- **C3 - calibracao proxy do score**: reescala operacional do score aprendido;
- **SHAP / interpretabilidade**: leitura dos drivers do score da rede.

Linguagem metodologica prudente:
- o score nnPU deve ser interpretado como um **score PU associado a positividade latente**;
- trata-se de uma **ordenacao da propensao latente** sob as hipoteses do modelo;
- nao se deve ler esta trilha como recuperacao perfeita da verdade latente.


In [ ]:
# Setup reprodutivel e validacoes de ambiente
import os
import random
import sys
import time
import platform
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as exc:
    raise ImportError(
        "PyTorch (torch) nao esta instalado. Instale com `uv add torch` e rode `uv sync`."
    ) from exc

try:
    import shap
    SHAP_AVAILABLE = True
    shap_import_error = None
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    shap_import_error = str(exc)

SEED = 42
SMOKE = False # True
SMOKE_MAX_ROWS = 300_000

RUN_C1_MAIN = True
RUN_C2_PRIOR_GRID = True
RUN_C3_CALIBRATION = True
RUN_SHAP = True

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"
OUTPUT_DIR = ROOT / "dados" / "dados_processados" / "outputs_pu"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo ausente: {DATA_PATH}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Ambiente:")
print(f"- Python: {sys.version.split()[0]}")
print(f"- Plataforma: {platform.platform()}")
print(f"- PyTorch: {torch.__version__}")
print(f"- CUDA disponivel: {torch.cuda.is_available()}")
print(f"- Device: {DEVICE}")
print(f"- Seed: {SEED}")
print(f"- SMOKE: {SMOKE}")
print(f"- SHAP disponivel: {SHAP_AVAILABLE}")
if not SHAP_AVAILABLE:
    print(f"- aviso SHAP: {shap_import_error}")


# 2. Carregamento dos dados

Carregamento do consolidado em nivel pessoa. Em `SMOKE=True`, aplicamos amostragem
reprodutivel para teste rapido de ponta a ponta.

Observacao:
- resultados de `SMOKE` servem para validar a pipeline;
- nao substituem a execucao completa para conclusoes finais do artigo.


In [ ]:
t0 = time.perf_counter()
df = pd.read_parquet(DATA_PATH, engine="pyarrow")
print(f"Shape original: {df.shape}")
print(f"Leitura: {time.perf_counter()-t0:.2f}s")

if SMOKE and len(df) > SMOKE_MAX_ROWS:
    df = df.sample(n=SMOKE_MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"[SMOKE] Shape: {df.shape}")
    print(f"[SMOKE] Seed da amostragem: {SEED}")

if "s" not in df.columns:
    raise ValueError("Coluna 's' ausente.")

display(
    pd.concat([
        df["s"].value_counts(dropna=False).rename("count"),
        (df["s"].value_counts(dropna=False) / len(df)).rename("pct"),
    ], axis=1).sort_index()
)


# 3. Configuracao de modelagem

Definicao de colunas especiais, hiperparametros do modelo neural, grade de prior,
flags de execucao e parametros dos blocos opcionais C2, C3 e SHAP.


In [ ]:
TARGET_COL = "s"
GROUP_COL = "id_familia"
ID_COLS = ["cpf_agricultor", "id_familia"]
META_COLS = ["ibge"]

PI_P_MAIN = 0.73
PI_P_GRID = [0.65, 0.70, 0.73, 0.75]

BATCH_SIZE = 4096
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 3
DROPOUT = 0.10

SENSITIVITY_EPOCHS = 8 if SMOKE else 6
SENSITIVITY_PATIENCE = 2

CALIBRATION_METHOD = "platt"
CALIBRATION_FIT_ON = "val"
CALIBRATION_APPLY_ON = "test"
CALIBRATION_USE_SCORE = "score_corrigido"
CALIBRATION_MIN_POS = 100
CALIBRATION_SAVE_ARTIFACTS = True
CALIBRATION_ISOTONIC_OUT_OF_BOUNDS = "clip"
CALIBRATION_MAX_ITERS = 1000
TEMPERATURE_INIT = 1.0
TEMPERATURE_MAX_ITERS = 300
TEMPERATURE_LR = 1e-2
TEMPERATURE_WEIGHT_DECAY = 0.0

SHAP_METHOD = "gradient"
SHAP_BACKGROUND_SIZE = 256
SHAP_EXPLAIN_SIZE = 1000 if not SMOKE else 256
SHAP_RANDOM_STATE = SEED
SHAP_SAMPLE_SOURCE = "test"
SAVE_SHAP_FIGURES = True

print("Configuracao pronta.")
print(f"- PI_P_MAIN: {PI_P_MAIN}")
print(f"- PI_P_GRID: {PI_P_GRID}")
print(f"- RUN_C2_PRIOR_GRID: {RUN_C2_PRIOR_GRID}")
print(f"- RUN_C3_CALIBRATION: {RUN_C3_CALIBRATION}")
print(f"- RUN_SHAP: {RUN_SHAP}")


# 4. Preparacao das features

- `ibge` fica fora das features.
- `SimpleImputer(median)` para missing simples e reprodutivel.
- `StandardScaler` para estabilizar treino da rede.
- Mantemos a estrutura densa e numerica do notebook original.


In [ ]:
required = set([TARGET_COL, GROUP_COL] + ID_COLS + META_COLS)
miss = sorted(required - set(df.columns))
if miss:
    raise ValueError(f"Colunas obrigatorias ausentes: {miss}")

s_num = pd.to_numeric(df[TARGET_COL], errors="coerce")
if s_num.isna().any() or (~s_num.isin([0, 1])).any():
    raise ValueError("s deve ser binaria {0,1}.")
df[TARGET_COL] = s_num.astype("int8")

feature_cols = [c for c in df.columns if c not in ([TARGET_COL] + ID_COLS + META_COLS)]
if any(c in feature_cols for c in META_COLS):
    raise ValueError("META_COLS nao podem entrar no modelo.")

non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
if non_numeric:
    raise ValueError(f"Features nao numericas detectadas: {non_numeric[:10]}")

num_cols = list(feature_cols)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

print(f"Features no modelo: {len(num_cols)}")
if len(num_cols) != 43:
    print(f"[aviso] Esperado ~43, encontrado {len(num_cols)}")

display(df[num_cols].isna().mean().sort_values(ascending=False).head(15).to_frame("pct_missing"))


# 5. Split por grupos

`GroupShuffleSplit` em 2 etapas para 80/10/10 com bloqueio por `id_familia`,
evitando vazamento entre familia no treino/val/test.


In [ ]:
X_all = df[num_cols]
y_all = df[TARGET_COL].astype("int8")
g_all = df[GROUP_COL].astype("string")
if g_all.isna().any():
    raise ValueError("id_familia com NA.")

s1 = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=SEED)
idx_train, idx_tmp = next(s1.split(X_all, y=y_all, groups=g_all))

s2 = GroupShuffleSplit(n_splits=1, train_size=0.5, test_size=0.5, random_state=SEED)
idx_val_rel, idx_test_rel = next(
    s2.split(idx_tmp, y=y_all.iloc[idx_tmp], groups=g_all.iloc[idx_tmp])
)
idx_val = idx_tmp[idx_val_rel]
idx_test = idx_tmp[idx_test_rel]

ov_tv = len(np.intersect1d(pd.unique(g_all.iloc[idx_train]), pd.unique(g_all.iloc[idx_val])))
ov_tt = len(np.intersect1d(pd.unique(g_all.iloc[idx_train]), pd.unique(g_all.iloc[idx_test])))
ov_vt = len(np.intersect1d(pd.unique(g_all.iloc[idx_val]), pd.unique(g_all.iloc[idx_test])))
if not (ov_tv == ov_tt == ov_vt == 0):
    raise AssertionError("Vazamento entre grupos detectado.")

X_train_raw, X_val_raw, X_test_raw = X_all.iloc[idx_train], X_all.iloc[idx_val], X_all.iloc[idx_test]
y_train = y_all.iloc[idx_train].to_numpy(dtype="int8", copy=False)
y_val = y_all.iloc[idx_val].to_numpy(dtype="int8", copy=False)
y_test = y_all.iloc[idx_test].to_numpy(dtype="int8", copy=False)

# Fit no treino para evitar leakage estatistico.
X_train = scaler.fit_transform(imputer.fit_transform(X_train_raw)).astype("float32")
X_val = scaler.transform(imputer.transform(X_val_raw)).astype("float32")
X_test = scaler.transform(imputer.transform(X_test_raw)).astype("float32")

print(f"Linhas train/val/test: {len(idx_train):,} / {len(idx_val):,} / {len(idx_test):,}")
print(f"Overlaps grupos: train-val={ov_tv}, train-test={ov_tt}, val-test={ov_vt}")


def split_dist(name, y, groups):
    y = pd.Series(y)
    return {
        "split": name,
        "n": int(len(y)),
        "n_familias": int(pd.Series(groups).nunique(dropna=False)),
        "s_positivos": int((y == 1).sum()),
        "s_unlabeled": int((y == 0).sum()),
        "pct_s1": float((y == 1).mean()),
    }


display(
    pd.DataFrame(
        [
            split_dist("train", y_train, g_all.iloc[idx_train]),
            split_dist("val", y_val, g_all.iloc[idx_val]),
            split_dist("test", y_test, g_all.iloc[idx_test]),
        ]
    )
)

# Metadados para saida auditavel de validacao e teste.
df_val_meta = (
    df.iloc[idx_val][["cpf_agricultor", "id_familia", "ibge", TARGET_COL]]
    .rename(columns={TARGET_COL: "s"})
    .reset_index(drop=True)
)
df_test_meta = (
    df.iloc[idx_test][["cpf_agricultor", "id_familia", "ibge", TARGET_COL]]
    .rename(columns={TARGET_COL: "s"})
    .reset_index(drop=True)
)


# 6. C1 - Class prior fixo

`PI_P_MAIN` e a **hipotese operacional principal** desta execucao da Trilha C.

Ponto metodologico importante:
- `PI_P_MAIN` nao deve ser tratado como verdade definitiva;
- ele e um prior de trabalho para o fluxo principal;
- a robustez da conclusao precisa ser lida a luz de **C2 - Sensibilidade ao prior**.


In [ ]:
if not RUN_C1_MAIN:
    raise ValueError("Esta notebook foi organizada para executar C1 como fluxo principal.")

if not (0.0 < PI_P_MAIN < 1.0):
    raise ValueError("PI_P_MAIN invalido.")
if any((p <= 0.0 or p >= 1.0) for p in PI_P_GRID):
    raise ValueError("PI_P_GRID deve conter valores em (0,1).")

priors_to_run = list(PI_P_GRID) if RUN_C2_PRIOR_GRID else []

print(f"Prior principal (C1): {PI_P_MAIN}")
print(f"Priors da sensibilidade (C2): {priors_to_run}")
display(pd.DataFrame({"prior": PI_P_GRID}))


# 7. Definicao do modelo neural

MLP pequena para tabular: `input -> 64 -> 32 -> 1 (logit)`, com ReLU e dropout.

A arquitetura permanece simples por comparabilidade, legibilidade e facilidade de auditoria.


In [ ]:
class MLPBinaryClassifier(nn.Module):
    # Modelo intencionalmente simples para comparabilidade e legibilidade.
    def __init__(self, input_dim: int, dropout: float = 0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


print(MLPBinaryClassifier(input_dim=len(num_cols), dropout=DROPOUT))


# 8. Definicao da loss nnPU

Usamos risco nao-negativo (Kiryo et al., 2017):
- risco positivo: `pi_p * E_P[l(g(x))]`
- risco negativo estimado: `E_U[l(-g(x))] - pi_p * E_P[l(-g(x))]`
- correcao nnPU: `max(0, risco_negativo)`

Este e o nucleo metodologico da Trilha C.
O clipping do risco negativo e o diferencial do nnPU para evitar comportamento degenerado
e overfitting do risco empirico.


In [ ]:
def nnpu_non_negative_loss(logits_pos: torch.Tensor, logits_unl: torch.Tensor, pi_p: float):
    # Loss logistica base; evita assumir que U e negativo puro.
    if logits_pos.numel() == 0 or logits_unl.numel() == 0:
        raise ValueError("Batches P/U vazios.")

    loss_pos_as_pos = torch.nn.functional.softplus(-logits_pos)
    loss_pos_as_neg = torch.nn.functional.softplus(logits_pos)
    loss_unl_as_neg = torch.nn.functional.softplus(logits_unl)

    positive_risk = pi_p * loss_pos_as_pos.mean()
    negative_risk_unclipped = loss_unl_as_neg.mean() - pi_p * loss_pos_as_neg.mean()
    negative_risk = torch.clamp(negative_risk_unclipped, min=0.0)
    total_risk = positive_risk + negative_risk

    diag = {
        "positive_risk": float(positive_risk.detach().cpu()),
        "negative_risk_unclipped": float(negative_risk_unclipped.detach().cpu()),
        "negative_risk_clipped": float(negative_risk.detach().cpu()),
        "negative_risk_was_clipped": bool(float(negative_risk_unclipped.detach().cpu()) < 0.0),
    }
    return total_risk, diag


def recall_at_fraction(y_true: np.ndarray, scores: np.ndarray, frac: float):
    if len(y_true) != len(scores):
        raise ValueError("Tamanhos incompativeis.")
    k = max(1, int(np.ceil(frac * len(y_true))))
    top = np.argsort(-scores)[:k]
    total_pos = int((y_true == 1).sum())
    hit_pos = int((y_true[top] == 1).sum())
    return k, hit_pos, (hit_pos / total_pos if total_pos > 0 else 0.0)


def precision_at_fraction(y_true: np.ndarray, scores: np.ndarray, frac: float):
    if len(y_true) != len(scores):
        raise ValueError("Tamanhos incompativeis.")
    k = max(1, int(np.ceil(frac * len(y_true))))
    top = np.argsort(-scores)[:k]
    hit_pos = int((y_true[top] == 1).sum())
    return k, hit_pos, (hit_pos / k if k > 0 else 0.0)


def predict_scores_sigmoid(model: nn.Module, X: np.ndarray, device: torch.device, batch_size: int = 8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i : i + batch_size]).to(device)
            out.append(torch.sigmoid(model(xb)).detach().cpu().numpy())
    return np.concatenate(out).astype("float32", copy=False) if out else np.array([], dtype="float32")


def predict_logits(model: nn.Module, X: np.ndarray, device: torch.device, batch_size: int = 8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i : i + batch_size]).to(device)
            out.append(model(xb).detach().cpu().numpy())
    return np.concatenate(out).astype("float32", copy=False) if out else np.array([], dtype="float32")


def make_pu_dataloaders(X_train_arr: np.ndarray, y_train_arr: np.ndarray, batch_size: int, seed: int):
    X_pos_train = X_train_arr[y_train_arr == 1]
    X_unl_train = X_train_arr[y_train_arr == 0]
    if len(X_pos_train) == 0 or len(X_unl_train) == 0:
        raise ValueError("Treino precisa ter P e U.")

    gen_pos = torch.Generator(); gen_pos.manual_seed(seed)
    gen_unl = torch.Generator(); gen_unl.manual_seed(seed + 1)

    pos_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_pos_train)),
        batch_size=min(batch_size, len(X_pos_train)),
        shuffle=True,
        generator=gen_pos,
        drop_last=False,
    )
    unl_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_unl_train)),
        batch_size=min(batch_size, len(X_unl_train)),
        shuffle=True,
        generator=gen_unl,
        drop_last=False,
    )
    return pos_loader, unl_loader, len(X_pos_train), len(X_unl_train)


def evaluate_scores_with_direction(
    y_true: np.ndarray,
    score_model: np.ndarray,
):
    mean_s1 = float(np.mean(score_model[y_true == 1])) if np.any(y_true == 1) else np.nan
    mean_s0 = float(np.mean(score_model[y_true == 0])) if np.any(y_true == 0) else np.nan
    inv = bool(mean_s1 < mean_s0)
    score_corr = 1.0 - score_model if inv else score_model

    auc = float(roc_auc_score(y_true, score_corr))
    _, _, r1 = recall_at_fraction(y_true, score_corr, 0.01)
    _, _, r5 = recall_at_fraction(y_true, score_corr, 0.05)
    _, _, r10 = recall_at_fraction(y_true, score_corr, 0.10)
    _, _, p1 = precision_at_fraction(y_true, score_corr, 0.01)
    _, _, p5 = precision_at_fraction(y_true, score_corr, 0.05)
    _, _, p10 = precision_at_fraction(y_true, score_corr, 0.10)

    return {
        "score_corrigido": score_corr.astype("float32", copy=False),
        "AUC_s": float(auc),
        "Recall@1%": float(r1),
        "Recall@5%": float(r5),
        "Recall@10%": float(r10),
        "Precision@1%": float(p1),
        "Precision@5%": float(p5),
        "Precision@10%": float(p10),
        "score_invertido": bool(inv),
        "mean_score_s1": float(mean_s1),
        "mean_score_s0": float(mean_s0),
    }


def apply_score_direction(score_model: np.ndarray, invert: bool) -> np.ndarray:
    return (1.0 - score_model if invert else score_model).astype("float32", copy=False)


def overlap_top_fraction(scores_a: np.ndarray, scores_b: np.ndarray, frac: float) -> float:
    k = max(1, int(np.ceil(frac * len(scores_a))))
    top_a = set(np.argsort(-scores_a)[:k].tolist())
    top_b = set(np.argsort(-scores_b)[:k].tolist())
    return float(len(top_a & top_b) / max(len(top_a), 1))


def overlap_top_n(scores_a: np.ndarray, scores_b: np.ndarray, n: int) -> float:
    n = min(n, len(scores_a), len(scores_b))
    top_a = set(np.argsort(-scores_a)[:n].tolist())
    top_b = set(np.argsort(-scores_b)[:n].tolist())
    return float(len(top_a & top_b) / max(n, 1))


# 9. Treinamento do modelo (C1)

Treino principal do nnPU com DataLoaders separados de P/U, Adam, e early stopping
por `AUC_s` na validacao.


In [ ]:
def train_nnpu_once(
    X_train_arr: np.ndarray,
    y_train_arr: np.ndarray,
    X_val_arr: np.ndarray,
    y_val_arr: np.ndarray,
    *,
    pi_p: float,
    epochs: int,
    lr: float,
    weight_decay: float,
    batch_size: int,
    patience: int,
    seed: int,
    dropout: float,
    device: torch.device,
    verbose: bool = True,
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = MLPBinaryClassifier(input_dim=X_train_arr.shape[1], dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_loader, unl_loader, n_pos_train, n_unl_train = make_pu_dataloaders(
        X_train_arr, y_train_arr, batch_size, seed
    )
    steps = min(len(pos_loader), len(unl_loader))
    if steps <= 0:
        raise ValueError("Passos por epoca invalidos.")

    history = []
    best_auc = -np.inf
    best_epoch = -1
    best_state = None
    rounds_no_improve = 0
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        e0 = time.perf_counter()
        model.train()
        run_loss = 0.0
        run_pr = 0.0
        run_nr = 0.0
        run_nr_unclipped = 0.0
        run_nr_clipped_hits = 0

        pos_it = iter(pos_loader)
        unl_it = iter(unl_loader)
        for _ in range(steps):
            xb_pos = next(pos_it)[0].to(device)
            xb_unl = next(unl_it)[0].to(device)

            optimizer.zero_grad(set_to_none=True)
            logits_pos = model(xb_pos)
            logits_unl = model(xb_unl)
            loss, diag = nnpu_non_negative_loss(logits_pos, logits_unl, pi_p)
            loss.backward()
            optimizer.step()

            run_loss += float(loss.detach().cpu())
            run_pr += float(diag["positive_risk"])
            run_nr += float(diag["negative_risk_clipped"])
            run_nr_unclipped += float(diag["negative_risk_unclipped"])
            run_nr_clipped_hits += int(diag["negative_risk_was_clipped"])

        train_loss = run_loss / steps
        train_pr = run_pr / steps
        train_nr = run_nr / steps
        train_nr_unclipped = run_nr_unclipped / steps
        train_nr_clipped_share = run_nr_clipped_hits / steps

        val_score_model = predict_scores_sigmoid(model, X_val_arr, device)
        val_diag = evaluate_scores_with_direction(y_val_arr, val_score_model)
        auc_val = float(val_diag["AUC_s"])
        r10_val = float(val_diag["Recall@10%"])

        epoch_s = time.perf_counter() - e0
        history.append(
            {
                "epoch": int(epoch),
                "train_loss": float(train_loss),
                "train_positive_risk": float(train_pr),
                "train_negative_risk": float(train_nr),
                "train_negative_risk_unclipped": float(train_nr_unclipped),
                "train_negative_risk_clipped_share": float(train_nr_clipped_share),
                "val_auc_s": float(auc_val),
                "val_recall_at_10pct": float(r10_val),
                "val_score_invertido": bool(val_diag["score_invertido"]),
                "epoch_seconds": float(epoch_s),
            }
        )

        metric = auc_val if not np.isnan(auc_val) else -np.inf
        if metric > best_auc + 1e-7:
            best_auc = metric
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            rounds_no_improve = 0
        else:
            rounds_no_improve += 1

        if verbose:
            print(
                f"Epoch {epoch:02d} | loss={train_loss:.6f} | "
                f"val_auc_s={auc_val:.6f} | val_recall@10%={r10_val:.6f} | s={epoch_s:.2f}"
            )

        if rounds_no_improve >= patience:
            if verbose:
                print(f"Early stopping na epoca {epoch}.")
            break

    total_s = time.perf_counter() - t0
    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = int(history[-1]["epoch"])
    model.load_state_dict(best_state)

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "train_seconds": float(total_s),
        "best_epoch": int(best_epoch),
        "best_auc_val": float(best_auc),
        "epochs_executadas": int(len(history)),
        "n_pos_train": int(n_pos_train),
        "n_unl_train": int(n_unl_train),
        "device_usado": str(device),
    }


train_result = train_nnpu_once(
    X_train,
    y_train,
    X_val,
    y_val,
    pi_p=PI_P_MAIN,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    patience=PATIENCE,
    seed=SEED,
    dropout=DROPOUT,
    device=DEVICE,
    verbose=True,
)

model = train_result["model"]
training_history = train_result["history"].copy()
train_seconds = float(train_result["train_seconds"])
best_epoch = int(train_result["best_epoch"])
best_auc_val = float(train_result["best_auc_val"])
epochs_executadas = int(train_result["epochs_executadas"])


# 10. Diagnostico do treinamento (C1)

Relato de convergencia, historico por epoca e componentes do risco nnPU.


In [ ]:
print("Diagnostico do treino principal (C1):")
print(f"- epocas executadas: {epochs_executadas}")
print(f"- melhor epoca: {best_epoch}")
print(f"- melhor AUC_s (val): {best_auc_val:.6f}")
print(f"- tempo treino: {train_seconds:.2f}s")
print(f"- device: {train_result['device_usado']}")
print(f"- n_pos_train: {train_result['n_pos_train']:,}")
print(f"- n_unl_train: {train_result['n_unl_train']:,}")
display(training_history)


# 11. Geracao de scores (C1)

Geramos logits e scores em validacao e teste. A validacao sera reutilizada em C3,
enquanto o teste sustenta a leitura principal de desempenho operacional.


In [ ]:
logits_val = predict_logits(model, X_val, DEVICE)
logits_test = predict_logits(model, X_test, DEVICE)

score_model_val = (1.0 / (1.0 + np.exp(-logits_val))).astype("float32", copy=False)
score_model_test = (1.0 / (1.0 + np.exp(-logits_test))).astype("float32", copy=False)

df_scores_val = df_val_meta.copy()
df_scores_val["score_model"] = score_model_val
df_scores_val["score_corrigido"] = df_scores_val["score_model"].astype("float32")

df_scores_test = df_test_meta.copy()
df_scores_test["score_model"] = score_model_test
df_scores_test["score_corrigido"] = df_scores_test["score_model"].astype("float32")

print(df_scores_test.shape)
display(df_scores_test["score_model"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]))


# 12. Diagnostico do score (C1)

Sanity check de direcao do score. O evento de inversao, quando existir, deve ficar explicito
no fluxo e nos artefatos, e nao escondido como detalhe silencioso.


In [ ]:
val_diag = evaluate_scores_with_direction(y_val, df_scores_val["score_model"].to_numpy(dtype="float32", copy=False))
score_invertido = bool(val_diag["score_invertido"])

if score_invertido:
    df_scores_val["score_corrigido"] = (1.0 - df_scores_val["score_model"]).astype("float32")
    df_scores_test["score_corrigido"] = (1.0 - df_scores_test["score_model"]).astype("float32")
else:
    df_scores_val["score_corrigido"] = df_scores_val["score_model"].astype("float32")
    df_scores_test["score_corrigido"] = df_scores_test["score_model"].astype("float32")

test_mean_s1 = float(df_scores_test.loc[df_scores_test["s"] == 1, "score_corrigido"].mean())
test_mean_s0 = float(df_scores_test.loc[df_scores_test["s"] == 0, "score_corrigido"].mean())

df_score_direction_summary = pd.DataFrame(
    [
        {
            "trilha": "C",
            "modelo": "nnPU MLP (PyTorch)",
            "score_invertido": bool(score_invertido),
            "direcao_final": "score_corrigido = 1 - score_model" if score_invertido else "score_corrigido = score_model",
            "mean_score_val_s1": float(val_diag["mean_score_s1"]),
            "mean_score_val_s0": float(val_diag["mean_score_s0"]),
            "mean_score_test_s1": float(test_mean_s1),
            "mean_score_test_s0": float(test_mean_s0),
        }
    ]
)

print("Diagnostico de direcao do score (C1):")
display(df_score_direction_summary)
print("Distribuicao de score_corrigido por s (teste):")
display(df_scores_test.groupby("s")["score_corrigido"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))


# 13. Avaliacao do modelo (C1)

Avaliacao operacional em `s` observado (proxy):
- `AUC_s`
- `Recall@1%`
- `Recall@5%`
- `Recall@10%`

Limitacao importante:
estas metricas nao validam diretamente `y` latente.


In [ ]:
y_test_eval = y_test.astype("int8", copy=False)
score_eval = df_scores_test["score_corrigido"].to_numpy(dtype="float64", copy=False)

AUC_s = float(roc_auc_score(y_test_eval, score_eval))
print(f"AUC_s (proxy): {AUC_s:.6f}")

rows = []
for frac in [0.01, 0.05, 0.10]:
    k_abs, hit_pos, rec = recall_at_fraction(y_test_eval, score_eval, frac)
    rows.append(
        {
            "k_frac": float(frac),
            "k_absoluto": int(k_abs),
            "positivos_topk": int(hit_pos),
            "positivos_totais_test": int((y_test_eval == 1).sum()),
            "recall_at_k": float(rec),
        }
    )

df_recall_k = pd.DataFrame(rows)
display(df_recall_k)


# 14. Ranking de unlabeled (C1)

O ranking representa unlabeled com maior score PU, nao observacao direta da classe latente.
Trata-se de um instrumento de priorizacao analitica e operacional.


In [ ]:
ranking_unlabeled = (
    df_scores_test[df_scores_test["s"] == 0]
    .sort_values("score_corrigido", ascending=False)
    .reset_index(drop=True)
)

TOP_N_RANKING = 100_000
ranking_unlabeled_top100k = ranking_unlabeled.head(TOP_N_RANKING).copy()
print(ranking_unlabeled_top100k.shape)
display(ranking_unlabeled_top100k.head(10))


# 15. C2 - Sensibilidade ao prior

Esta secao e **crucial** para interpretar a Trilha C.

O nnPU depende do prior. Portanto:
- C1 deve ser lido como execucao principal sob `PI_P_MAIN`;
- C2 mostra o quanto os resultados mudam quando alteramos `pi_p`;
- conclusoes robustas exigem estabilidade razoavel de ranking e metricas.


In [ ]:
main_score_test = df_scores_test["score_corrigido"].to_numpy(dtype="float32", copy=False)

prior_rows = []
if RUN_C2_PRIOR_GRID:
    for i, prior in enumerate(priors_to_run):
        print(f"[C2] prior={prior:.2f}")
        sens = train_nnpu_once(
            X_train,
            y_train,
            X_val,
            y_val,
            pi_p=float(prior),
            epochs=SENSITIVITY_EPOCHS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            batch_size=BATCH_SIZE,
            patience=SENSITIVITY_PATIENCE,
            seed=SEED + 100 + i,
            dropout=DROPOUT,
            device=DEVICE,
            verbose=False,
        )

        score_val_local = predict_scores_sigmoid(sens["model"], X_val, device=DEVICE)
        score_test_local = predict_scores_sigmoid(sens["model"], X_test, device=DEVICE)

        val_local_diag = evaluate_scores_with_direction(y_val, score_val_local)
        local_invert = bool(val_local_diag["score_invertido"])
        score_test_local_corr = apply_score_direction(score_test_local, local_invert)
        score_test_local_corr64 = score_test_local_corr.astype("float64", copy=False)

        hist_local = sens["history"].copy()
        prior_rows.append(
            {
                "prior": float(prior),
                "AUC_s": float(roc_auc_score(y_test_eval, score_test_local_corr64)),
                "Recall@1%": float(recall_at_fraction(y_test_eval, score_test_local_corr64, 0.01)[2]),
                "Recall@5%": float(recall_at_fraction(y_test_eval, score_test_local_corr64, 0.05)[2]),
                "Recall@10%": float(recall_at_fraction(y_test_eval, score_test_local_corr64, 0.10)[2]),
                "best_epoch": int(sens["best_epoch"]),
                "epochs_executadas": int(sens["epochs_executadas"]),
                "train_seconds": float(sens["train_seconds"]),
                "score_invertido": bool(local_invert),
                "mean_score_s1": float(np.mean(score_test_local_corr[y_test_eval == 1])),
                "mean_score_s0": float(np.mean(score_test_local_corr[y_test_eval == 0])),
                "freq_negative_risk_unclipped_lt_zero": float((hist_local["train_negative_risk_unclipped"] < 0).mean()),
                "mean_positive_risk": float(hist_local["train_positive_risk"].mean()),
                "mean_negative_risk_clipped": float(hist_local["train_negative_risk"].mean()),
                "overlap_top1pct_vs_main": float(overlap_top_fraction(main_score_test, score_test_local_corr, 0.01)),
                "overlap_top5pct_vs_main": float(overlap_top_fraction(main_score_test, score_test_local_corr, 0.05)),
                "overlap_top10pct_vs_main": float(overlap_top_fraction(main_score_test, score_test_local_corr, 0.10)),
                "overlap_top1000_vs_main": float(overlap_top_n(main_score_test, score_test_local_corr, 1000)),
                "overlap_top5000_vs_main": float(overlap_top_n(main_score_test, score_test_local_corr, 5000)),
            }
        )

prior_sensitivity_summary = pd.DataFrame(prior_rows).sort_values("prior").reset_index(drop=True)
if prior_sensitivity_summary.empty:
    prior_sensitivity_summary = pd.DataFrame(
        columns=[
            "prior", "AUC_s", "Recall@1%", "Recall@5%", "Recall@10%", "best_epoch",
            "epochs_executadas", "train_seconds", "score_invertido", "mean_score_s1",
            "mean_score_s0", "freq_negative_risk_unclipped_lt_zero", "mean_positive_risk",
            "mean_negative_risk_clipped", "overlap_top1pct_vs_main", "overlap_top5pct_vs_main",
            "overlap_top10pct_vs_main", "overlap_top1000_vs_main", "overlap_top5000_vs_main"
        ]
    )

display(prior_sensitivity_summary)


# 16. C3 - Calibracao proxy do score

Esta secao implementa uma **reescala/calibracao operacional do score** aprendido pelo nnPU.

Ponto metodologico:
- a calibracao aqui e **proxy em `s`**, nao calibracao perfeita de `P(y=1|x)`;
- serve para avaliar se uma reescala melhora ou degrada a leitura operacional do score;
- C3 e opcional na execucao, mas obrigatoria como secao parametrizada da notebook.


In [ ]:
def fit_temperature_scaler(
    logits_val: np.ndarray,
    y_val_arr: np.ndarray,
    *,
    init_temperature: float,
    lr: float,
    max_iters: int,
    weight_decay: float,
    device: torch.device,
):
    temp = torch.nn.Parameter(torch.tensor([init_temperature], dtype=torch.float32, device=device))
    optimizer = torch.optim.Adam([temp], lr=lr, weight_decay=weight_decay)
    logits_tensor = torch.from_numpy(logits_val.astype("float32")).to(device)
    y_tensor = torch.from_numpy(y_val_arr.astype("float32")).to(device)

    for _ in range(max_iters):
        optimizer.zero_grad(set_to_none=True)
        scaled_logits = logits_tensor / torch.clamp(temp, min=1e-3)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(scaled_logits, y_tensor)
        loss.backward()
        optimizer.step()

    learned_temp = float(torch.clamp(temp.detach().cpu(), min=1e-3).item())
    return learned_temp


calibration_status = "not_run"
calibration_message = "C3 desativado."
calibration_artifact = None
df_calibration_summary = pd.DataFrame()

if RUN_C3_CALIBRATION:
    calibration_status = "skipped"
    calibration_message = "Condicoes minimas nao atendidas."

    fit_y = y_val.copy() if CALIBRATION_FIT_ON == "val" else y_test.copy()
    apply_y = y_test.copy() if CALIBRATION_APPLY_ON == "test" else y_val.copy()

    fit_df = df_scores_val.copy() if CALIBRATION_FIT_ON == "val" else df_scores_test.copy()
    apply_df = df_scores_test.copy() if CALIBRATION_APPLY_ON == "test" else df_scores_val.copy()

    fit_score = pd.to_numeric(fit_df[CALIBRATION_USE_SCORE], errors="coerce").fillna(0.0).to_numpy(dtype="float64")
    apply_score = pd.to_numeric(apply_df[CALIBRATION_USE_SCORE], errors="coerce").fillna(0.0).to_numpy(dtype="float64")

    if int((fit_y == 1).sum()) >= CALIBRATION_MIN_POS:
        try:
            if CALIBRATION_METHOD == "platt":
                calibrator = LogisticRegression(max_iter=CALIBRATION_MAX_ITERS, random_state=SEED)
                calibrator.fit(fit_score.reshape(-1, 1), fit_y)
                score_fit_cal = calibrator.predict_proba(fit_score.reshape(-1, 1))[:, 1]
                score_apply_cal = calibrator.predict_proba(apply_score.reshape(-1, 1))[:, 1]
                calibration_artifact = {"method": "platt", "model": calibrator}
            elif CALIBRATION_METHOD == "isotonic":
                calibrator = IsotonicRegression(out_of_bounds=CALIBRATION_ISOTONIC_OUT_OF_BOUNDS)
                calibrator.fit(fit_score, fit_y)
                score_fit_cal = calibrator.transform(fit_score)
                score_apply_cal = calibrator.transform(apply_score)
                calibration_artifact = {"method": "isotonic", "model": calibrator}
            elif CALIBRATION_METHOD == "temperature":
                oriented_logits_val = -logits_val if score_invertido else logits_val
                oriented_logits_test = -logits_test if score_invertido else logits_test
                fit_logits = oriented_logits_val if CALIBRATION_FIT_ON == "val" else oriented_logits_test
                apply_logits = oriented_logits_test if CALIBRATION_APPLY_ON == "test" else oriented_logits_val
                learned_temp = fit_temperature_scaler(
                    logits_val=fit_logits,
                    y_val_arr=fit_y.astype("int8", copy=False),
                    init_temperature=TEMPERATURE_INIT,
                    lr=TEMPERATURE_LR,
                    max_iters=TEMPERATURE_MAX_ITERS,
                    weight_decay=TEMPERATURE_WEIGHT_DECAY,
                    device=DEVICE,
                )
                score_fit_cal = 1.0 / (1.0 + np.exp(-(fit_logits / learned_temp)))
                score_apply_cal = 1.0 / (1.0 + np.exp(-(apply_logits / learned_temp)))
                calibration_artifact = {"method": "temperature", "temperature": learned_temp}
            else:
                raise ValueError(
                    "CALIBRATION_METHOD deve ser um entre {'platt', 'isotonic', 'temperature'}."
                )

            calibration_status = "ok"
            calibration_message = "Calibracao executada."
        except Exception as exc:
            calibration_status = "failed"
            calibration_message = str(exc)
            score_fit_cal = fit_score.copy()
            score_apply_cal = apply_score.copy()
    else:
        score_fit_cal = fit_score.copy()
        score_apply_cal = apply_score.copy()
        calibration_message = (
            f"Positivos insuficientes para calibracao: {(fit_y == 1).sum()} < {CALIBRATION_MIN_POS}"
        )


In [ ]:
if RUN_C3_CALIBRATION:
    raw_fit_auc = float(roc_auc_score(fit_y, fit_score))
    raw_apply_auc = float(roc_auc_score(apply_y, apply_score))
    cal_fit_auc = float(roc_auc_score(fit_y, score_fit_cal))
    cal_apply_auc = float(roc_auc_score(apply_y, score_apply_cal))

    raw_fit_r1 = float(recall_at_fraction(fit_y, fit_score, 0.01)[2])
    raw_fit_r5 = float(recall_at_fraction(fit_y, fit_score, 0.05)[2])
    raw_fit_r10 = float(recall_at_fraction(fit_y, fit_score, 0.10)[2])
    raw_apply_r1 = float(recall_at_fraction(apply_y, apply_score, 0.01)[2])
    raw_apply_r5 = float(recall_at_fraction(apply_y, apply_score, 0.05)[2])
    raw_apply_r10 = float(recall_at_fraction(apply_y, apply_score, 0.10)[2])

    cal_fit_r1 = float(recall_at_fraction(fit_y, score_fit_cal, 0.01)[2])
    cal_fit_r5 = float(recall_at_fraction(fit_y, score_fit_cal, 0.05)[2])
    cal_fit_r10 = float(recall_at_fraction(fit_y, score_fit_cal, 0.10)[2])
    cal_apply_r1 = float(recall_at_fraction(apply_y, score_apply_cal, 0.01)[2])
    cal_apply_r5 = float(recall_at_fraction(apply_y, score_apply_cal, 0.05)[2])
    cal_apply_r10 = float(recall_at_fraction(apply_y, score_apply_cal, 0.10)[2])

    df_calibration_summary = pd.DataFrame(
        [
            {"stage": CALIBRATION_FIT_ON, "score_variant": "bruto", "AUC_s": raw_fit_auc, "Recall@1%": raw_fit_r1, "Recall@5%": raw_fit_r5, "Recall@10%": raw_fit_r10},
            {"stage": CALIBRATION_FIT_ON, "score_variant": "calibrado", "AUC_s": cal_fit_auc, "Recall@1%": cal_fit_r1, "Recall@5%": cal_fit_r5, "Recall@10%": cal_fit_r10},
            {"stage": CALIBRATION_APPLY_ON, "score_variant": "bruto", "AUC_s": raw_apply_auc, "Recall@1%": raw_apply_r1, "Recall@5%": raw_apply_r5, "Recall@10%": raw_apply_r10},
            {"stage": CALIBRATION_APPLY_ON, "score_variant": "calibrado", "AUC_s": cal_apply_auc, "Recall@1%": cal_apply_r1, "Recall@5%": cal_apply_r5, "Recall@10%": cal_apply_r10},
        ]
    )

    if CALIBRATION_APPLY_ON == "test":
        df_scores_test["score_calibrado"] = pd.Series(score_apply_cal).astype("float32")
    else:
        df_scores_val["score_calibrado"] = pd.Series(score_apply_cal).astype("float32")

if "score_calibrado" not in df_scores_test.columns:
    df_scores_test["score_calibrado"] = df_scores_test["score_corrigido"].astype("float32")

if df_calibration_summary.empty:
    df_calibration_summary = pd.DataFrame(
        [
            {
                "stage": CALIBRATION_APPLY_ON,
                "score_variant": "bruto",
                "AUC_s": float(AUC_s),
                "Recall@1%": float(df_recall_k.loc[df_recall_k["k_frac"] == 0.01, "recall_at_k"].iloc[0]),
                "Recall@5%": float(df_recall_k.loc[df_recall_k["k_frac"] == 0.05, "recall_at_k"].iloc[0]),
                "Recall@10%": float(df_recall_k.loc[df_recall_k["k_frac"] == 0.10, "recall_at_k"].iloc[0]),
            }
        ]
    )

if calibration_status == "ok":
    bruto_apply = df_calibration_summary[
        (df_calibration_summary["stage"] == CALIBRATION_APPLY_ON)
        & (df_calibration_summary["score_variant"] == "bruto")
    ].iloc[0]
    calibrado_apply = df_calibration_summary[
        (df_calibration_summary["stage"] == CALIBRATION_APPLY_ON)
        & (df_calibration_summary["score_variant"] == "calibrado")
    ].iloc[0]

    if calibrado_apply["AUC_s"] > bruto_apply["AUC_s"] + 1e-6:
        calibration_conclusion = "A calibracao agregou valor em AUC_s no conjunto de aplicacao."
    elif calibrado_apply["AUC_s"] < bruto_apply["AUC_s"] - 1e-6:
        calibration_conclusion = "A calibracao degradou a AUC_s no conjunto de aplicacao."
    else:
        calibration_conclusion = "A calibracao teve efeito neutro ou marginal em AUC_s."
else:
    calibration_conclusion = "A calibracao nao foi executada com sucesso; mantem-se o score bruto corrigido."

print("Resumo C3:")
print(f"- status: {calibration_status}")
print(f"- mensagem: {calibration_message}")
print(f"- metodo: {CALIBRATION_METHOD}")
print(f"- conclusao: {calibration_conclusion}")
display(df_calibration_summary)


# 17. SHAP / interpretabilidade

SHAP e usado aqui como ferramenta de interpretabilidade do score aprendido pela rede.

Leitura correta:
- SHAP explica variacoes do score da Trilha C;
- a interpretacao continua condicionada as hipoteses do modelo nnPU;
- a validacao empirica segue proxy em `s`.


In [ ]:
df_shap_feature_importance = pd.DataFrame()
df_shap_top_unlabeled_importance = pd.DataFrame()
shap_status = "not_run"
shap_message = "Bloco SHAP desativado."
shap_effective_method = None

def _normalize_shap_values(raw_values):
    if isinstance(raw_values, list):
        raw_values = raw_values[0]
    raw_values = np.asarray(raw_values, dtype="float64")
    if raw_values.ndim == 3 and raw_values.shape[-1] == 1:
        raw_values = raw_values[..., 0]
    elif raw_values.ndim == 3:
        raw_values = raw_values[:, :, 0]
    elif raw_values.ndim == 1:
        raw_values = raw_values.reshape(1, -1)
    return raw_values

def _predict_score_for_shap(matrix):
    matrix = np.asarray(matrix, dtype="float32")
    tensor = torch.from_numpy(matrix).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor).detach().cpu().numpy().reshape(-1)
    return 1.0 / (1.0 + np.exp(-logits))

if RUN_SHAP:
    if not SHAP_AVAILABLE:
        shap_status = "unavailable"
        shap_message = f"SHAP indisponivel no ambiente: {shap_import_error}"
        print(shap_message)
    else:
        try:
            background_size = min(SHAP_BACKGROUND_SIZE, len(X_train))
            explain_size = min(SHAP_EXPLAIN_SIZE, len(X_test))
            rng = np.random.default_rng(SHAP_RANDOM_STATE)

            background_idx = rng.choice(len(X_train), size=background_size, replace=False)
            background_np = X_train[background_idx].astype("float32", copy=False)
            explain_np = X_test[:explain_size].astype("float32", copy=False)

            model.eval()
            shap_values = None
            explainer = None

            if SHAP_METHOD in {"deep", "gradient"}:
                try:
                    background_tensor = torch.from_numpy(background_np).to(DEVICE)
                    explain_tensor = torch.from_numpy(explain_np).to(DEVICE)
                    if SHAP_METHOD == "deep":
                        explainer = shap.DeepExplainer(model, background_tensor)
                    else:
                        explainer = shap.GradientExplainer(model, background_tensor)
                    shap_values = explainer.shap_values(explain_tensor)
                    shap_effective_method = SHAP_METHOD
                except Exception as exc:
                    warnings.warn(
                        f"Falha no {SHAP_METHOD} explainer: {exc}. Tentando KernelExplainer."
                    )

            if shap_values is None:
                kernel_background_size = min(64 if SMOKE else 128, len(background_np))
                kernel_explain_size = min(64 if SMOKE else 128, len(explain_np))
                kernel_background = background_np[:kernel_background_size]
                explain_np = explain_np[:kernel_explain_size]
                explainer = shap.KernelExplainer(_predict_score_for_shap, kernel_background)
                shap_values = explainer.shap_values(
                    explain_np,
                    nsamples=min(128, 2 * explain_np.shape[1] + 1),
                )
                shap_effective_method = "kernel"

            shap_values = _normalize_shap_values(shap_values)

            df_shap_feature_importance = pd.DataFrame(
                {
                    "feature": num_cols,
                    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
                    "mean_shap": shap_values.mean(axis=0),
                }
            ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_values,
                explain_np,
                feature_names=num_cols,
                plot_type="bar",
                show=False,
            )
            plt.title(f"Trilha C - SHAP bar summary ({shap_effective_method})")
            plt.tight_layout()
            if SAVE_SHAP_FIGURES:
                plt.show()
            else:
                plt.close()

            try:
                plt.figure(figsize=(10, 6))
                shap.summary_plot(
                    shap_values,
                    explain_np,
                    feature_names=num_cols,
                    show=False,
                )
                plt.title(f"Trilha C - SHAP beeswarm ({shap_effective_method})")
                plt.tight_layout()
                if SAVE_SHAP_FIGURES:
                    plt.show()
                else:
                    plt.close()
            except Exception as exc:
                warnings.warn(f"Falha ao gerar beeswarm SHAP: {exc}")

            n_top = min(SHAP_EXPLAIN_SIZE, len(ranking_unlabeled_top100k))
            if n_top > 0:
                top_cpfs = ranking_unlabeled_top100k["cpf_agricultor"].head(n_top).tolist()
                test_cpfs = df_test_meta["cpf_agricultor"].astype(str).tolist()
                pos_map = {cpf: i for i, cpf in enumerate(test_cpfs)}
                top_idx = [pos_map[str(cpf)] for cpf in top_cpfs if str(cpf) in pos_map]
                if top_idx:
                    top_np = X_test[np.array(top_idx)].astype("float32", copy=False)
                    if shap_effective_method == "kernel":
                        top_np = top_np[: min(len(top_np), 64 if SMOKE else 128)]
                        top_shap_values = explainer.shap_values(
                            top_np,
                            nsamples=min(128, 2 * top_np.shape[1] + 1),
                        )
                    else:
                        top_tensor = torch.from_numpy(top_np).to(DEVICE)
                        top_shap_values = explainer.shap_values(top_tensor)

                    top_shap_values = _normalize_shap_values(top_shap_values)
                    df_shap_top_unlabeled_importance = pd.DataFrame(
                        {
                            "feature": num_cols,
                            "mean_abs_shap": np.abs(top_shap_values).mean(axis=0),
                            "mean_shap": top_shap_values.mean(axis=0),
                        }
                    ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)

            shap_status = "ok"
            shap_message = f"SHAP executado com sucesso via {shap_effective_method}."
        except Exception as exc:
            shap_status = "failed"
            shap_message = str(exc)
            print(f"Falha no bloco SHAP: {shap_message}")

print("Resumo SHAP:")
print(f"- status: {shap_status}")
print(f"- metodo efetivo: {shap_effective_method}")
print(f"- mensagem: {shap_message}")
if not df_shap_feature_importance.empty:
    display(df_shap_feature_importance.head(20))
if not df_shap_top_unlabeled_importance.empty:
    print("Top ranking unlabeled - importancia SHAP:")
    display(df_shap_top_unlabeled_importance.head(20))


# 18. Salvamento de artefatos

Salvar scores, ranking, historico, sensibilidade ao prior, calibracao e interpretabilidade
com prefixo `trilhaC` e sufixo `_smoke` quando aplicavel.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
suffix = "_smoke" if SMOKE else ""

scores_path = OUTPUT_DIR / f"trilhaC_scores_test{suffix}.parquet"
ranking_path = OUTPUT_DIR / f"trilhaC_ranking_unlabeled_top100k{suffix}.parquet"
metrics_path = OUTPUT_DIR / f"trilhaC_metrics_summary{suffix}.parquet"
history_path = OUTPUT_DIR / f"trilhaC_training_history{suffix}.parquet"
prior_path = OUTPUT_DIR / f"trilhaC_prior_sensitivity_summary{suffix}.parquet"
direction_path = OUTPUT_DIR / f"trilhaC_score_direction_summary{suffix}.parquet"
calibration_path = OUTPUT_DIR / f"trilhaC_calibration_summary{suffix}.parquet"
shap_path = OUTPUT_DIR / f"trilhaC_shap_feature_importance{suffix}.parquet"
shap_top_path = OUTPUT_DIR / f"trilhaC_shap_top_unlabeled_importance{suffix}.parquet"
feature_names_path = OUTPUT_DIR / f"trilhaC_feature_names{suffix}.parquet"
xtest_path = OUTPUT_DIR / f"trilhaC_X_test_transformado{suffix}.parquet"
model_path = OUTPUT_DIR / f"trilhaC_model_nnpu{suffix}.pt"
manifest_path = OUTPUT_DIR / f"trilhaC_artifacts_manifest{suffix}.parquet"

r1 = float(df_recall_k.loc[df_recall_k["k_frac"] == 0.01, "recall_at_k"].iloc[0])
r5 = float(df_recall_k.loc[df_recall_k["k_frac"] == 0.05, "recall_at_k"].iloc[0])
r10 = float(df_recall_k.loc[df_recall_k["k_frac"] == 0.10, "recall_at_k"].iloc[0])

metrics_summary = pd.DataFrame(
    [
        {
            "trilha": "C",
            "modelo": "nnPU MLP (PyTorch)",
            "AUC_s": float(AUC_s),
            "Recall@1%": float(r1),
            "Recall@5%": float(r5),
            "Recall@10%": float(r10),
            "prior_principal": float(PI_P_MAIN),
            "score_invertido": bool(score_invertido),
            "train_seconds": float(train_seconds),
            "epochs_executadas": int(epochs_executadas),
            "best_epoch": int(best_epoch),
            "device_usado": str(train_result["device_usado"]),
        }
    ]
)

required = {"cpf_agricultor", "ibge", "s", "score_model", "score_corrigido", "score_calibrado"}
if not required.issubset(df_scores_test.columns):
    raise ValueError(f"Colunas ausentes em df_scores_test: {sorted(required - set(df_scores_test.columns))}")


In [ ]:
df_feature_names = pd.DataFrame(
    {
        "ordem_feature": np.arange(len(num_cols), dtype="int32"),
        "feature": num_cols,
    }
)

df_X_test_transformado = pd.DataFrame(X_test, columns=num_cols)
df_X_test_transformado.insert(0, "s", df_test_meta["s"].astype("int8"))
df_X_test_transformado.insert(0, "ibge", df_test_meta["ibge"].astype("string"))
df_X_test_transformado.insert(0, "id_familia", df_test_meta["id_familia"].astype("string"))
df_X_test_transformado.insert(0, "cpf_agricultor", df_test_meta["cpf_agricultor"].astype("string"))

torch.save(
    {
        "state_dict": model.state_dict(),
        "input_dim": int(len(num_cols)),
        "dropout": float(DROPOUT),
        "pi_p_main": float(PI_P_MAIN),
        "score_invertido": bool(score_invertido),
    },
    model_path,
)


In [ ]:
df_scores_test.to_parquet(scores_path, index=False)
ranking_unlabeled_top100k.to_parquet(ranking_path, index=False)
metrics_summary.to_parquet(metrics_path, index=False)
training_history.to_parquet(history_path, index=False)
prior_sensitivity_summary.to_parquet(prior_path, index=False)
df_score_direction_summary.to_parquet(direction_path, index=False)
if CALIBRATION_SAVE_ARTIFACTS:
    df_calibration_summary.to_parquet(calibration_path, index=False)
df_feature_names.to_parquet(feature_names_path, index=False)
df_X_test_transformado.to_parquet(xtest_path, index=False)

if not df_shap_feature_importance.empty:
    df_shap_feature_importance.to_parquet(shap_path, index=False)
if not df_shap_top_unlabeled_importance.empty:
    df_shap_top_unlabeled_importance.to_parquet(shap_top_path, index=False)

df_manifest = pd.DataFrame(
    [
        {
            "trilha": "C",
            "seed": int(SEED),
            "smoke": bool(SMOKE),
            "smoke_max_rows": int(SMOKE_MAX_ROWS),
            "n_train": int(len(X_train)),
            "n_val": int(len(X_val)),
            "n_test": int(len(X_test)),
            "n_features": int(len(num_cols)),
            "prior_principal": float(PI_P_MAIN),
            "score_invertido": bool(score_invertido),
            "device_usado": str(train_result["device_usado"]),
            "calibration_status": calibration_status,
            "shap_status": shap_status,
            "scores_file": scores_path.name,
            "ranking_file": ranking_path.name,
            "metrics_file": metrics_path.name,
            "history_file": history_path.name,
            "prior_file": prior_path.name,
            "direction_file": direction_path.name,
            "calibration_file": calibration_path.name,
            "feature_names_file": feature_names_path.name,
            "xtest_file": xtest_path.name,
            "model_file": model_path.name,
        }
    ]
)
df_manifest.to_parquet(manifest_path, index=False)

print(scores_path)
print(ranking_path)
print(metrics_path)
print(history_path)
print(prior_path)
print(direction_path)
print(calibration_path)
print(feature_names_path)
print(xtest_path)
print(model_path)
print(manifest_path)
display(metrics_summary)


# 19. Comparacao com trilhas anteriores

Comparacao executiva entre:
- Trilha A = proxy supervisionado em `s`
- Trilha B = PU classico
- Trilha C = nnPU / risk-based

Se artefatos externos estiverem ausentes, o notebook nao deve quebrar.


In [ ]:
comparison_rows = []

row_a = {
    "trilha": "Trilha A",
    "modelo": "Proxy supervisionado",
    "paradigma": "Supervisionado em s (proxy)",
    "AUC_s": np.nan,
    "Recall@1%": np.nan,
    "Recall@5%": np.nan,
    "Recall@10%": np.nan,
}
for p in [
    OUTPUT_DIR / "trilhaA_metricas_resumo.parquet",
    OUTPUT_DIR / "trilhaA_metricas_resumo_smoke.parquet",
]:
    if p.exists():
        df_a = pd.read_parquet(p, engine="pyarrow")
        if "componente" in df_a.columns:
            row = df_a[df_a["componente"] == "A4_score_final"]
            if row.empty:
                row = df_a.tail(1)
        else:
            row = df_a.tail(1)
        row = row.iloc[0]
        row_a.update(
            {
                "AUC_s": float(row.get("auc_s_test", row.get("AUC_s", np.nan))),
                "Recall@1%": float(row.get("recall_at_1pct_test", row.get("Recall@1%", np.nan))),
                "Recall@5%": float(row.get("recall_at_5pct_test", row.get("Recall@5%", np.nan))),
                "Recall@10%": float(row.get("recall_at_10pct_test", row.get("Recall@10%", np.nan))),
            }
        )
        break

row_b = {
    "trilha": "Trilha B",
    "modelo": "PU classico",
    "paradigma": "PU classico sob SCAR",
    "AUC_s": np.nan,
    "Recall@1%": np.nan,
    "Recall@5%": np.nan,
    "Recall@10%": np.nan,
}
for p in [
    OUTPUT_DIR / "trilhaB_comparacao_interna.parquet",
    OUTPUT_DIR / "trilhaB_comparacao_interna_smoke.parquet",
]:
    if p.exists():
        df_b = pd.read_parquet(p, engine="pyarrow")
        sort_auc_col = "auc_s_test" if "auc_s_test" in df_b.columns else (
            "auc_proxy_s" if "auc_proxy_s" in df_b.columns else None
        )
        sort_recall_col = "recall_at_5pct_test" if "recall_at_5pct_test" in df_b.columns else (
            "recall_at_5pct" if "recall_at_5pct" in df_b.columns else None
        )
        sort_cols = [c for c in [sort_auc_col, sort_recall_col] if c is not None]
        if sort_cols:
            df_b = df_b.sort_values(
                sort_cols,
                ascending=[False] * len(sort_cols),
            )
        row = df_b.iloc[0]
        row_b.update(
            {
                "modelo": str(row.get("model_name", "PU classico")),
                "AUC_s": float(row.get("auc_s_test", row.get("auc_proxy_s", np.nan))),
                "Recall@1%": float(row.get("recall_at_1pct_test", row.get("recall_at_1pct", np.nan))),
                "Recall@5%": float(row.get("recall_at_5pct_test", row.get("recall_at_5pct", np.nan))),
                "Recall@10%": float(row.get("recall_at_10pct_test", row.get("recall_at_10pct", np.nan))),
            }
        )
        break


In [ ]:
row_c = {
    "trilha": "Trilha C",
    "modelo": "nnPU",
    "paradigma": "PU por risco nao-negativo (nnPU)",
    "AUC_s": float(AUC_s),
    "Recall@1%": float(r1),
    "Recall@5%": float(r5),
    "Recall@10%": float(r10),
}

comparison_rows.extend([row_a, row_b, row_c])
df_comparacao_trilhas = pd.DataFrame(comparison_rows)
display(df_comparacao_trilhas)

best_prev_auc = float(
    df_comparacao_trilhas[df_comparacao_trilhas["trilha"] != "Trilha C"]["AUC_s"].max()
)
if np.isnan(best_prev_auc):
    print("Nao foi possivel calcular ganho de AUC_s vs trilhas anteriores por ausencia de artefatos.")
else:
    print(
        "Ganho AUC_s da Trilha C vs melhor anterior disponivel: "
        f"{(AUC_s - best_prev_auc):+.6f}"
    )


# 20. Conclusao da trilha

Sintese final da Trilha C:
- resultado principal de C1;
- sensibilidade observada em C2;
- efeito ou nao da calibracao em C3;
- principais achados do SHAP;
- limitacoes por avaliar em `s`;
- papel metodologico desta trilha no artigo.


In [ ]:
print("Resumo final - Trilha C (nnPU)")
print(f"- AUC_s: {AUC_s:.6f}")
print(f"- Recall@1%: {r1:.6f}")
print(f"- Recall@5%: {r5:.6f}")
print(f"- Recall@10%: {r10:.6f}")
print(f"- prior principal (C1): {PI_P_MAIN}")
print(f"- score_invertido: {score_invertido}")
print(f"- train_seconds: {train_seconds:.2f}")
print(f"- epochs_executadas: {epochs_executadas}")
print(f"- best_epoch: {best_epoch}")
print(f"- device_usado: {train_result['device_usado']}")
if not prior_sensitivity_summary.empty:
    print("- C2 mostrou a sensibilidade do nnPU ao prior e a estabilidade relativa do ranking.")
print(f"- C3 status: {calibration_status}")
print(f"- C3 conclusao: {calibration_conclusion}")
print(f"- SHAP status: {shap_status}")
print("- A Trilha C e metodologicamente central para a familia moderna de PU learning.")
print("- A robustez empirica precisa ser lida a luz da analise de prior e da limitacao de avaliar em s.")
